In [ ]:
# ============ IMPORTS ============
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import mlflow
from mlflow.models import infer_signature
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier

# ============ CONFIGURATION ============
feature_set = "confirmed" # "all" or "confirmed" -  features confirmed by boruta algorithm
load_hyperparameters = True
window = 7
xgboost_device = "cuda" # change to "cpu" if no GPU available

# Fallback parameters — used only if load_hyperparameters = False
fallback_params = {
    "baseline_stratified" : {},
    "baseline_buy_hold" : {},
    "log_reg":        {"max_iter": 1000},
    "random_forest":  {"n_estimators": 200, "max_depth": 8},
    "svc":            {"kernel": "rbf"},
    "xgboost":        {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05},
}

bounded_features_and_limits = {
    "RSI":      (0,    100),
    "STOCHRSI": (0,    100),
    "STOCH_K":  (0,    100),
    "STOCH_D":  (0,    100),
    "WILLR":    (-100, 0),
    "MFI":      (0,    100),
    "ADX":      (0,    100),
    "ADXR":     (0,    100),
    "PLUS_DI":  (0,    100),
    "MINUS_DI": (0,    100),
    "AROONOSC": (-100, 100),
    "ULTOSC":   (0,    100),
    "CMO":      (-100, 100),
}
# =======================================

# ============ LOAD LABELLED DATA ============
labelled_data_cache = pd.read_pickle('data_cache/04_labelled_data.pkl')
labelled_df = labelled_data_cache['labelled_df']
X_train = labelled_data_cache['X_train']
X_test = labelled_data_cache['X_test']
y_train = labelled_data_cache['y_train']
y_test = labelled_data_cache['y_test']
profit_target = labelled_data_cache['profit_target']
stop_loss = labelled_data_cache['stop_loss']

# ============ SCALE LABELLED DATA ============
unbounded_feature_cols = [col for col in X_train.columns if col not in bounded_features_and_limits]
robust_scaler = RobustScaler().fit(X_train[unbounded_feature_cols])
X_train[unbounded_feature_cols] = robust_scaler.transform(X_train[unbounded_feature_cols])
X_test[unbounded_feature_cols]  = robust_scaler.transform(X_test[unbounded_feature_cols])

# ============ LABEL REMAPPING ============
label_map   = {-1: 0, 0: 1, 1: 2}
label_unmap = { 0: -1, 1: 0, 2: 1}
y_train_enc = y_train.map(label_map)

# =========== LOAD LATEST FEATURE SELECTION FROM MLFLOW ============
fs_experiment  = mlflow.get_experiment_by_name("05_feature_selection")
# List Previous Runs
fs_runs = mlflow.search_runs(
            experiment_ids=[fs_experiment.experiment_id],
            filter_string = "status = 'FINISHED'",
            order_by = ['start_time DESC'])
# Load latest run ID
fs_run_id = fs_runs.iloc[0]["run_id"]
# Load artifact
fs_artifact_path = mlflow.artifacts.download_artifacts(
    run_id = fs_run_id, 
    artifact_path="05_feature_status.pkl"
)
# Load dataframe
fs_status = pd.read_pickle(fs_artifact_path)
# Load features
confirmed_features = fs_status.loc[fs_status['status'] == 'confirmed', 'feature'].tolist()
if feature_set == "confirmed":
    X_train = X_train[confirmed_features]
    X_test  = X_test[confirmed_features]

# ============ LOAD BEST HYPERPARAMETERS FROM MLFLOW ============
model_params = {name: dict(p) for name, p in fallback_params.items()}
hp_trials = {}

if load_hyperparameters:
    hp_experiment = mlflow.get_experiment_by_name("06_hyperparameter_tuning")
    # List Previous Runs
    hp_runs = mlflow.search_runs(
        experiment_ids=[hp_experiment.experiment_id],
        # filter_string="tags.Hyperpar_optimised_for = 'mcc'", # use tags.Hyperpar_optimised_for = 'GT_Score'/'F1_macro'
        order_by=["start_time DESC"] # start_time pr metrics.best_objective
    )
    for model_name in model_params:
        run_name_filter = f"hyperparameters_{model_name}_{feature_set}_features"
        model_runs = hp_runs[hp_runs["tags.mlflow.runName"] == run_name_filter]

        if model_runs.empty:
            print(f"No tuning run found for {model_name}_{feature_set} — using fallback params.")
            continue

        best_run_id = model_runs.iloc[0]["run_id"]
        best_params_artifact_path = mlflow.artifacts.download_artifacts(
            run_id=best_run_id,
            artifact_path=f"06_hp_best_params_{model_name}_{feature_set}.pkl"
        )
        hp_trials_artifact_path = mlflow.artifacts.download_artifacts(
            run_id=best_run_id,
            artifact_path=f"06_hp_trials_{model_name}_{feature_set}.pkl"
        )
        params = pd.read_pickle(best_params_artifact_path).to_dict()
        # Integer arguments not stored as integer, so convert
        int_params = {"n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_iter"}
        params = {parameter: int(value) if parameter in int_params else value for parameter, value in params.items()}
        
        model_params[model_name] = params
        hp_trials[model_name] = pd.read_pickle(hp_trials_artifact_path).to_dict()

# ============ BUILD MODELS ============
models = {
    # # ── Baselines ──────────────────────────────────────────
    "baseline_stratified": DummyClassifier(strategy="stratified", random_state=42),
    "baseline_buy_hold" : DummyClassifier(strategy="constant", constant=1),
    # ── ML Models ──────────────────────────────────────────
    "log_reg": LogisticRegression(**model_params["log_reg"],
                                    solver="saga", max_iter=2000,
                                    class_weight="balanced", random_state=42),
    "svc": SVC(**model_params["svc"],
                kernel="rbf",probability=True,
                class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(**model_params["random_forest"],
                                            class_weight="balanced",
                                            random_state=42,
                                            n_jobs=-1),
    "xgboost": XGBClassifier(**model_params["xgboost"],
                            learning_rate=0.05,
                            random_state=42,
                            tree_method="hist",
                            device=xgboost_device,
                            n_jobs=1)
}

In [ ]:
# =========== MLFLOW SETUP ============
mlflow.set_experiment("07_triple_barrier_classification") # or "bitcoin_near_peak_classification"
mlflow.sklearn.autolog()  # auto-log params, metrics, model, etc.

# ============ TRAIN MODELS ============
for model_name, model in models.items():
    with mlflow.start_run(run_name=f"{model_name}_{feature_set}_features"):
        # Train model
        if model_name == "xgboost": # XGBoost needs integer classes
            model.fit(X_train, y_train_enc)
            train_pred = pd.Series(model.predict(X_train),index=X_train.index).map(label_unmap)
            test_pred = pd.Series(model.predict(X_test),index=X_test.index).map(label_unmap)
        else:
            model.fit(X_train, y_train)
            train_pred = pd.Series(model.predict(X_train), index=X_train.index)
            test_pred  = pd.Series(model.predict(X_test),  index=X_test.index)
            
# ============ LOG TRAINED MODEL ============
        signature = infer_signature(X_train, train_pred)
        # defines the expected input and output formats for ML model, for reliability
        
        model_info = mlflow.sklearn.log_model(
            model, 
            artifact_path="model",
            signature=signature
        )

# ============ LOG X/y_train and test metadata ============
        # Log X/y_train as mlflow input
        model_train_data = X_train.join(y_train).copy()
        mlflow.log_input(
            mlflow.data.from_pandas(
                model_train_data,  # single DataFrame with features + labels
                name="model_train_data",
                targets=y_train.name    # tells MLflow which column is the label
            ),
            context="final model training"
        )
        
        # Log X/y_test as mlflow input
        model_test_data = X_test.join(y_test).copy()
        mlflow.log_input(
            mlflow.data.from_pandas(
                model_test_data,  # single DataFrame with features + labels
                name="model_test_data",
                targets=y_test.name    # tells MLflow which column is the label
            ),
            context="final model testing"
        )

# ============ GENERATE PREDICTIONS ============
        # Train predictions
        train_proba = model.predict_proba(X_train)
        model_train_data[f'pred_{model_name}'] = train_pred
        model_train_data[f'proba_{model_name}'] = list(train_proba)  # Store as list of arrays
        # Test predictions
        test_proba = model.predict_proba(X_test)
        model_test_data[f'pred_{model_name}'] = test_pred
        model_test_data[f'proba_{model_name}'] = list(test_proba)
        
# ============ LOG FULL FEATURES, LABELS AND PREDICTIONS DATASET ============
        model_train_data.to_pickle(f'data_cache/07_final_ml_train_data_{model_name}_{feature_set}.pkl')
        mlflow.log_artifact(f'data_cache/07_final_ml_train_data_{model_name}_{feature_set}.pkl')
        model_test_data.to_pickle(f'data_cache/07_final_ml_test_data_{model_name}_{feature_set}.pkl')
        mlflow.log_artifact(f'data_cache/07_final_ml_test_data_{model_name}_{feature_set}.pkl')

# ============ LOG TRIPLE BARRIER PARAMETERS  ============
        mlflow.log_param("triple_barrier_profit_target", profit_target)
        mlflow.log_param("triple_barrier_stop_loss", stop_loss)
               
# ============ LOG FEATURES  ============
        # Triple barrier parameters affect which features get selected, so logging features
        for _, row in fs_status.iterrows():
            mlflow.log_param(f"feature_{row['feature']}", row['status'])
        
        if feature_set == "confirmed":
            final_ml_features = pd.Series(confirmed_features, name='feature')
        elif feature_set == "all":
            final_ml_features = fs_status['feature']
        
        final_ml_features.to_pickle(f'data_cache/07_final_ml_features_{model_name}_{feature_set}.pkl')
        mlflow.log_artifact(f'data_cache/07_final_ml_features_{model_name}_{feature_set}.pkl',artifact_path="features")